# Chapter 11 &mdash; Purely Right-Linear Grammars

**Concept 13 of the Chapter 11 decomposition:** *Purely Right-Linear Grammars*

At most one nonterminal per right-hand side, always at the right end &mdash; exactly the regular languages.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter11/Concept-Right-Linear-Grammars/Concept-Right-Linear-Grammars.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


A grammar is **right-linear** if every production has the form
$$A \to w \qquad \text{or} \qquad A \to wB$$
with $w\in\Sigma^*$ and $B\in N$: at most **one** nonterminal, always **rightmost**.

> **Right-linear grammars generate exactly the regular languages.**

Both directions are constructions you have already seen. $\Rightarrow$: the
nonterminal is the state, so the grammar *is* an NFA. $\Leftarrow$: Concept 12.

**Left**-linear grammars ($A \to Bw$) also give exactly the regular languages. It is
**mixing** the two that escapes regularity &mdash; Concept 15.

## 2. Definitions

### The CFG toolkit

A grammar is a dict; `language`, `nparses`, `parse_trees` and `leftmost` do the work.

In [ ]:
# --- a tiny CFG toolkit -------------------------------------------------
# A grammar is a dict with keys N (nonterminals), Sigma (terminals),
# S (start symbol) and P (productions: nonterminal -> list of RHS tuples).
# A right-hand side is a tuple of one-character symbols; () is epsilon.
# By convention UPPERCASE single letters are nonterminals.

def mkg(rules, start='S'):
    N = set(rules)
    P = {A: [tuple(r) for r in rhs] for A, rhs in rules.items()}
    Sigma = {c for rhs in P.values() for r in rhs for c in r if c not in N}
    return dict(N=N, Sigma=Sigma, S=start, P=P)

def show(G):
    print("N     =", sorted(G['N']))
    print("Sigma =", sorted(G['Sigma']))
    print("S     =", G['S'])
    for A in sorted(G['P']):
        alts = ' | '.join((''.join(r) if r else "''") for r in G['P'][A])
        print("   %s -> %s" % (A, alts))

def derivable(G, maxlen):
    # least fixed point: for each nonterminal, every terminal string of
    # length <= maxlen it derives.  Far cheaper than searching sentential
    # forms, and it terminates because the sets only grow and are bounded.
    T = {A: set() for A in G['N']}
    def spans(r):
        acc = {''}
        for x in r:
            src = T[x] if x in T else {x}
            acc = {a + b for a in acc for b in src if len(a) + len(b) <= maxlen}
            if not acc: break
        return acc
    changed = True
    while changed:
        changed = False
        for A in G['P']:
            for r in G['P'][A]:
                for w in spans(r):
                    if w not in T[A]:
                        T[A].add(w); changed = True
    return T

def language(G, maxlen):
    return sorted(derivable(G, maxlen)[G['S']], key=lambda s: (len(s), s))

def _spans(G, w, cap=None):
    # Bottom-up, shortest span first, so a span never depends on a LONGER
    # one.  Within a span we iterate |N|+1 times, which is enough to close
    # unit rules (A -> B) and epsilon rules.  Doing it top-down with a
    # "cycle guard" silently poisons the memo table, so we do not.
    n, N, P = len(w), G['N'], G['P']
    tab = {}                       # (A, i, j) -> count, or list of trees
    def get(sym, i, j):
        if sym not in N:
            if j == i + 1 and w[i] == sym:
                return 1 if cap is None else [sym]
            return 0 if cap is None else []
        return tab.get((sym, i, j), 0 if cap is None else [])
    def seqv(r, i, j):
        if not r:
            if i != j: return 0 if cap is None else []
            return 1 if cap is None else [()]
        acc = 0 if cap is None else []
        for k in range(i, j + 1):
            a = get(r[0], i, k)
            if not a: continue
            b = seqv(r[1:], k, j)
            if not b: continue
            if cap is None:
                acc += a * b
            else:
                for h in a:
                    for t in b:
                        acc.append((h,) + tuple(t))
                        if len(acc) >= cap: return acc
        return acc
    for length in range(0, n + 1):
        for i in range(0, n - length + 1):
            j = i + length
            for _ in range(len(N) + 1):
                grew = False
                for A in P:
                    v = []
                    for r in P[A]:
                        x = seqv(r, i, j)
                        if cap is None:
                            v.append(x)
                        else:
                            v += [(A,) + tuple(t) for t in x]
                            if len(v) >= cap: v = v[:cap]; break
                    v = sum(v) if cap is None else v
                    old = tab.get((A, i, j), 0 if cap is None else [])
                    if (v != old) if cap is None else (len(v) != len(old)):
                        tab[(A, i, j)] = v; grew = True
                if not grew: break
    return get(G['S'], 0, n)

def nparses(G, w):
    return _spans(G, w, cap=None)

def parse_trees(G, w, cap=8):
    return _spans(G, w, cap=cap)

def yield_of(t):
    return t if isinstance(t, str) else ''.join(yield_of(c) for c in t[1:])

def show_tree(t, ind=0):
    if isinstance(t, str):
        print("%s'%s'" % ('  ' * ind, t)); return
    print("%s%s" % ('  ' * ind, t[0]))
    for c in t[1:]: show_tree(c, ind + 1)

def leftmost(G, w):
    # the leftmost derivation read off one parse tree
    ts = parse_trees(G, w, cap=1)
    if not ts: return None
    steps, form = [], [G['S']]
    def expand(t, pos):
        # t is the subtree rooted at the nonterminal currently at `pos`
        if isinstance(t, str): return pos + 1
        kids = [c if isinstance(c, str) else c[0] for c in t[1:]]
        form[pos:pos+1] = kids
        steps.append(''.join(form) or "''")
        p = pos
        for c in t[1:]:
            p = expand(c, p)
        return p
    steps.append(G['S'])
    expand(ts[0], 0)
    return steps

### Recognising right-linearity

In [ ]:
def right_linear(G):
    for A, rhss in G['P'].items():
        for r in rhss:
            nts = [i for i, x in enumerate(r) if x in G['N']]
            if len(nts) > 1: return False
            if nts and nts[0] != len(r) - 1: return False
    return True

def left_linear(G):
    for A, rhss in G['P'].items():
        for r in rhss:
            nts = [i for i, x in enumerate(r) if x in G['N']]
            if len(nts) > 1: return False
            if nts and nts[0] != 0: return False
    return True

### Right-linear grammar to NFA: the nonterminal IS the state

In [ ]:
def rlg2nfa(G):
    lines = ['NFA']
    nm = {A: ('I' if A == G['S'] else 'S') + A for A in G['N']}
    fin = 'FEnd'
    for A, rhss in G['P'].items():
        for r in rhss:
            body = [x for x in r if x not in G['N']]
            tgt  = [x for x in r if x in G['N']]
            src, cur = nm[A], nm[A]
            for i, ch in enumerate(body):
                nxt = (nm[tgt[0]] if (i == len(body) - 1 and tgt)
                       else (fin if i == len(body) - 1 else 'M_%s_%d' % (A, i)))
                lines.append('%s : %s -> %s' % (cur, ch, nxt)); cur = nxt
            if not body:
                lines.append("%s : '' -> %s" % (src, nm[tgt[0]] if tgt else fin))
    lines.append("%s : '' -> %s" % (fin, fin))
    return md2mc('\n'.join(lines))

## 3. Tests

Classifying some grammars.

In [ ]:
Right = mkg({'A': ["0A", "1B", ""], 'B': ["0B", "1A"]}, 'A')
Left  = mkg({'A': ["A0", "B1", ""], 'B': ["B0", "A1"]}, 'A')
Mixed = mkg({'S': ["", "(A"], 'A': ["S)"]})
for name, G in [('Right', Right), ('Left', Left), ('Mixed', Mixed)]:
    print("%-6s right-linear %-6s left-linear %s"
          % (name, right_linear(G), left_linear(G)))
assert right_linear(Right) and not left_linear(Right)
assert left_linear(Left) and not right_linear(Left)
assert not right_linear(Mixed) and not left_linear(Mixed)

A right-linear grammar generates a **regular** language.

In [ ]:
N = rlg2nfa(Right)
D = min_dfa(nfa2dfa(N))
from itertools import product
L = set(language(Right, 8))
want = {''.join(p) for k in range(9) for p in product('01', repeat=k)
        if accepts_dfa(D, ''.join(p))}
print("grammar language == NFA language ?", L == want)
assert L == want
print("minimal DFA has %d states" % len(D["Q"]))

The round trip: DFA &rarr; right-linear grammar &rarr; NFA &rarr; DFA.

In [ ]:
import string
def dfa2cfg(D):
    order = sorted(D["Q"])
    nt = {q: string.ascii_uppercase[i] for i, q in enumerate(order)}
    rules = {}
    for q in order:
        alts = [a + nt[step_dfa(D, q, a)] for a in sorted(D["Sigma"])]
        if q in D["F"]: alts.append("")
        rules[nt[q]] = alts
    return mkg(rules, nt[D["q0"]])

D0 = md2mc('''DFA
IF : 0 -> Od
IF : 1 -> IF
Od : 0 -> IF
Od : 1 -> Od
''')
G  = dfa2cfg(D0)
assert right_linear(G)
D1 = min_dfa(nfa2dfa(rlg2nfa(G)))
print("original minimal %d states, round-tripped %d states"
      % (len(min_dfa(D0)["Q"]), len(D1["Q"])))
assert iso_dfa(min_dfa(D0), D1)
print("isomorphic :", iso_dfa(min_dfa(D0), D1))

**Left**-linear grammars give the regular languages too.

In [ ]:
Lg = mkg({'A': ["A0", "A1", "1"]}, 'A')
print("left-linear language :", language(Lg, 4))
assert left_linear(Lg)
assert all(w.startswith('1') for w in language(Lg, 4))
print("\n(it is the reverse of a right-linear one -- and regular languages are")
print(" closed under reversal, Chapter 7.)")

So the hierarchy has a bottom rung inside CFG.

In [ ]:
print("right-linear CFG  ==  left-linear CFG  ==  regular languages")
print("general CFG       ==  context-free languages, strictly larger")
print("\nConcept 15 shows that MIXING the two escapes the bottom rung.")

## 4. Animation

The DFA behind the right-linear grammar.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(min_dfa(nfa2dfa(rlg2nfa(mkg({'A': ['0A','1B',''], 'B': ['0B','1A']}, 'A')))), FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Convert a left-linear grammar to an NFA directly. What changes?
2. Why is the nonterminal "the same as" the state?
3. Write a right-linear grammar for "contains `101`".

In [ ]:
# Your work for the exercises above.